# SQLite Experiment Analytics

This notebook drives the **reproducible** `arrhythmia_ml` pipeline and writes
**real** sklearn metrics to SQLite.

Original exploratory notebooks are preserved under `notebooks_reference/`
and at the repo root — this file does **not** overwrite those results.

Variants: `baseline`, `pca`, `ros_pca` (train-only ROS), `smote_pca` (train-only SMOTE).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from arrhythmia_ml.config import ALL_VARIANTS, VARIANT_LABELS, RANDOM_STATE_SPLIT, RANDOM_STATE_MODEL
from arrhythmia_ml.preprocess import preprocess
from arrhythmia_ml.pipeline import run_variant
from arrhythmia_ml.db import init_db, create_experiment, persist_variant_outcome, finish_experiment
from arrhythmia_ml import analytics

DB_PATH = ROOT / "artifacts" / "experiments.db"
DATASET = ROOT / "Data" / "arrhythmia.csv"
print("Variants:")
for v in ALL_VARIANTS:
    print(f"  {v}: {VARIANT_LABELS[v]}")
print("seeds:", RANDOM_STATE_SPLIT, RANDOM_STATE_MODEL)

In [ ]:
X, y, meta = preprocess(DATASET)
print(meta)

conn = init_db(DB_PATH)
experiment_id = create_experiment(
    conn,
    run_name="sql_notebook_experiment",
    dataset_path=str(DATASET),
    n_samples=meta["n_samples"],
    n_features_raw=meta["n_features_raw"],
    notes="From sql_experiment_analytics.ipynb; train-only oversampling; fixed seeds",
)

for variant in ALL_VARIANTS:
    outcome = run_variant(
        X,
        y,
        variant=variant,
        imputer_strategy=meta["imputer_strategy"],
        drop_column_index=meta["drop_column_index"],
        random_state_split=RANDOM_STATE_SPLIT,
        random_state_model=RANDOM_STATE_MODEL,
    )
    config_id = persist_variant_outcome(conn, experiment_id, outcome)
    best = max(outcome["models"], key=lambda m: m["metrics"]["test"]["f1_weighted"])
    print(
        f"{VARIANT_LABELS[variant]} | config_id={config_id} | "
        f"best={best['model_name']} test_f1={best['metrics']['test']['f1_weighted']:.4f}"
    )

finish_experiment(conn, experiment_id)
print(f"Wrote experiment_id={experiment_id} -> {DB_PATH}")

In [ ]:
board = analytics.leaderboard(conn, experiment_id=experiment_id)
display(board.head(12))
display(analytics.best_model_per_variant(conn, experiment_id=experiment_id))
display(analytics.compare_model_across_variants(conn, model_name="Kernelized SVC", experiment_id=experiment_id))
display(analytics.pca_components_by_variant(conn, experiment_id=experiment_id))
conn.close()